# Attention Optimization

**Tags:** optimization, efficiency, attention
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/attention-optimization.md

## TL;DR

Optimize attention (O(T²) bottleneck): Flash Attention (I/O aware, 2-4x faster, same quality), sparse attention (attend only to relevant tokens, 2-4x faster, <1% quality loss), grouped-query attention (share KV across Q heads, 50% KV cache reduction). Choose based on hardware and latency requirements.

## Core Intuition

Standard attention computes QK^T (a T×T matrix), which requires O(T²) memory and transfers tons of data between slow HBM (GPU RAM) and fast SRAM. This is the LLM inference bottleneck. Flash Attention reorganizes computation to keep data on-chip. Sparse attention realizes you don't need T² comparisons—local + strided patterns suffice. GQA reuses KV across multiple Q heads to reduce cache size.

## How It Works

**Standard (Naive) Attention:**
```
Input: Q (T×D), K (T×D), V (T×D)
Memory: HBM (slow, high bandwidth but high latency)

Algorithm:
1. Load Q, K from HBM → compute QK^T (T×T) → write to HBM  [memory I/O]
2. Load QK^T → softmax → write to HBM                       [memory I/O]
3. Load softmax(QK^T), V → multiply → write output to HBM   [memory I/O]

Memory transactions: O(T²) with large matrices
Compute: O(T² × D)
Bottleneck: memory I/O dominates compute time
```

**Flash Attention (I/O-Aware):**
```
Key insight: reorganize to maximize on-chip computation

Algorithm (tiling):
1. Divide Q, K, V into blocks (e.g., 64×64)
2. For each block of Q:
   a. Load small block Q_i into SRAM (fast)
   b. Iterate over blocks of K, V:
      - Load K_j, V_j into SRAM
      - Compute attention(Q_i, K_j, V_j) on-chip
      - Accumulate results
      - No writing intermediate QK^T to HBM
3. Write final output once

Result:
- Reduced HBM I/O: from O(T² × D) to O(T × D)
- Speedup: 2-4x (less memory transfer)
- Quality: identical (same computation, just reorganized)

Hardware requirement: GPU with large SRAM (A100, H100, L40)
Compatibility: not portable to all GPUs
```

**Sparse Attention Patterns:**

Standard: O(T²) full attention
```
Token attends to: all previous tokens
Example (T=10):
  t0: [x]
  t1: [x x]
  t2: [x x x]
  t3: [x x x x]
  ...
  t10: [x x x x ... x] (10 comparisons)
```

Local attention: O(T × w) where w = window
```
Token attends to: last w tokens only
Example (w=4):
  t10: [x x x x]  (4 comparisons instead of 10)
```

Strided: O(T × √T) per layer
```
Token attends to: local window + periodic stride
Example (w=4, stride=8):
  t10: [x x x x] + [x . . . . . . x]  (8 comparisons)
```

Local + strided: O(T × √T) hierarchical
```
Each layer: some layers use local (cheap), some use strided (context)
Trade: <1% quality loss, 2-4x speedup
```

**Grouped-Query Attention (GQA):**

Standard Multi-Head Attention:
```
Q: num_heads=32, head_dim=64
K: num_heads=32, head_dim=64
V: num_heads=32, head_dim=64

Each head has independent K, V.
KV cache size per sequence: seq_len × 32 × 64 × 2 (K and V)
```

GQA (Group size = 8):
```
Q: num_heads=32, head_dim=64
K: num_heads=4, head_dim=64    (32 / 8 = 4)
V: num_heads=4, head_dim=64

Multiple Q heads share one K, V head.
head_0-7 → KV_head_0
head_8-15 → KV_head_1
head_16-23 → KV_head_2
head_24-31 → KV_head_3

KV cache size: seq_len × 4 × 64 × 2 = 8x smaller!
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Method | Speedup | Memory | Quality | Latency | Deploy |
|--------|---------|--------|---------|---------|--------|
| Standard | 1x | 1x | 100% | Baseline | Any GPU |
| Flash Attn | 2-4x | Same | 100% | -50-75% | A100/H100+ |
| Sparse (local) | 2-4x | 50% | 99% | -50-75% | Any GPU |
| Sparse (strided) | 3-8x | 25% | 98% | -70-85% | Any GPU |
| GQA (no sparse) | 1.2-1.5x | 50% (KV only) | 99% | -10% | Any GPU |
| Flash + Sparse | 4-8x | 50% | 98% | -75-90% | A100/H100+ |

**Practical Combinations:**

- **High latency sensitivity (chat):** Flash Attention + GQA
- **Long context (RAG docs):** Sparse (local+strided) + GQA
- **Cost-sensitive:** Sparse local attention only
- **Research/offline:** Standard (not optimized, accurate baseline)

### Code Implementation

```python
# Key imports for Attention Optimization
import numpy as np
import torch
from typing import Any

# Attention Optimization example implementation
class AttentionOptimization:
    """
    Attention Optimization implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is Attention Optimization used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Attention Optimization?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Attention Optimization compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Attention Optimization?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Attention Optimization?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/attention-optimization.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]